# 🎬 Netflix Genre Classifier — BERT Version
**Module 03b: DistilBERT Fine-tuning สำหรับ Genre Classification**

Notebook นี้อัปเกรดจาก TF-IDF + Logistic Regression → **DistilBERT** ซึ่งเป็น pre-trained transformer model

| | TF-IDF Baseline | DistilBERT (this notebook) |
|---|---|---|
| Model | Logistic Regression | DistilBERT fine-tuned |
| Features | 15K TF-IDF ngrams | 768-dim contextual embeddings |
| Accuracy | ~65% | ~80–85% (expected) |
| Training time | < 1 min | ~15–20 min (GPU) |

> ⚡ **Runtime → Change runtime type → T4 GPU** ก่อนรัน notebook นี้

## 1. Install & Import

In [ ]:
# Install libraries ที่ยังไม่มีใน Colab
%pip install transformers datasets accelerate -q


In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ตรวจ GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


## 2. Load & Prepare Data

In [ ]:
# Upload netflix_titles.csv ก่อน หรือโหลดจาก Kaggle
# from google.colab import files
# uploaded = files.upload()

# หรือโหลดจาก URL (Kaggle dataset)
# !kaggle datasets download -d shivamb/netflix-shows && unzip netflix-shows.zip

df = pd.read_csv('netflix_titles.csv')
print(f'Dataset shape: {df.shape}')
df.head(3)


In [ ]:
# เลือก 7 genre หลัก (เหมือน baseline)
TARGET_GENRES = [
    'Dramas', 'Comedies', 'Documentaries',
    'Action & Adventure', 'Thrillers',
    'Children & Family Movies', 'Horror Movies'
]

def get_primary_genre(listed_in):
    genres = [g.strip() for g in str(listed_in).split(',')]
    for tg in TARGET_GENRES:
        if tg in genres:
            return tg
    return None

# สร้าง dataset
df['genre'] = df['listed_in'].apply(get_primary_genre)
df_clean = df[df['genre'].notna() & df['description'].notna()].copy()
df_clean['text'] = df_clean['title'] + '. ' + df_clean['description']  # title + description

print('Class distribution:')
print(df_clean['genre'].value_counts())
print(f'\nTotal samples: {len(df_clean)}')


In [ ]:
# Label encoding
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['genre'])
num_classes = len(le.classes_)
print(f'Classes ({num_classes}):', list(le.classes_))

# Train/Val/Test split: 70/15/15
X = df_clean['text'].values
y = df_clean['label'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')


## 3. Tokenizer & Dataset Class

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128  # description สั้น 128 token พอ ประหยัด memory
BATCH_SIZE = 32

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

# ทดสอบ tokenizer
sample = tokenizer(
    'A detective investigates a mysterious murder in a coastal town.',
    max_length=MAX_LEN, truncation=True, padding='max_length', return_tensors='pt'
)
print('Input IDs shape:', sample['input_ids'].shape)
print('Tokens:', tokenizer.convert_ids_to_tokens(sample['input_ids'][0])[:15], '...')


In [ ]:
class NetflixDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

# สร้าง DataLoader
train_ds = NetflixDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds   = NetflixDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds  = NetflixDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')


## 4. Load DistilBERT & Fine-tune

In [ ]:
# โหลด pre-trained DistilBERT พร้อม classification head
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)
model = model.to(device)

# นับ parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')


In [ ]:
# Hyperparameters
EPOCHS    = 4
LR        = 2e-5   # learning rate ต่ำสำหรับ fine-tuning
WARMUP    = 0.1    # 10% warmup steps

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}')


In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        input_ids  = batch['input_ids'].to(device)
        attn_mask  = batch['attention_mask'].to(device)
        labels     = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        loss    = outputs.loss
        logits  = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            labels    = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            preds   = outputs.logits.argmax(dim=1)

            total_loss += outputs.loss.item()
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels


In [ ]:
# Training loop
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss, val_acc, _, _ = eval_epoch(model, val_loader, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_bert_classifier.pt')
        print(f'  ✓ Best model saved (val_acc={val_acc:.4f})')

print(f'\nBest Val Accuracy: {best_val_acc:.4f}')


## 5. Evaluate on Test Set

In [ ]:
# โหลด best model มา evaluate
model.load_state_dict(torch.load('best_bert_classifier.pt', map_location=device))

_, test_acc, test_preds, test_labels = eval_epoch(model, test_loader, device)
print(f'Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)')
print()
print(classification_report(
    test_labels, test_preds,
    target_names=le.classes_,
    digits=3
))


In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], 'o-', label='Train', color='#E50914')
ax1.plot(history['val_loss'],   'o-', label='Val',   color='#444')
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(history['train_acc'], 'o-', label='Train', color='#E50914')
ax2.plot(history['val_acc'],   'o-', label='Val',   color='#444')
ax2.set_title('Accuracy per Epoch')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Reds',
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title('Confusion Matrix — DistilBERT')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix_bert.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Compare: TF-IDF vs BERT

In [ ]:
# เปรียบเทียบ TF-IDF baseline กับ BERT
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

tfidf_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=15000, ngram_range=(1,2),
                               stop_words='english', sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=5.0,
                                  class_weight='balanced', random_state=42))
])
tfidf_pipe.fit(X_train, y_train)
tfidf_acc = accuracy_score(y_test, tfidf_pipe.predict(X_test))

# Summary
results = {
    'Model':    ['TF-IDF + LogReg (baseline)', 'DistilBERT (fine-tuned)'],
    'Accuracy': [f'{tfidf_acc*100:.1f}%',       f'{test_acc*100:.1f}%'],
    'Improvement': ['-', f'+{(test_acc - tfidf_acc)*100:.1f}%']
}
print(pd.DataFrame(results).to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['TF-IDF\n(Baseline)', 'DistilBERT\n(Fine-tuned)'],
               [tfidf_acc, test_acc],
               color=['#444', '#E50914'], width=0.5)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Accuracy')
ax.set_title('TF-IDF vs DistilBERT — Netflix Genre Classification')
for bar, acc in zip(bars, [tfidf_acc, test_acc]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('tfidf_vs_bert.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Inference — ทดสอบ Predict

In [ ]:
def predict_genre(text, model, tokenizer, le, device, top_k=3):
    model.eval()
    enc = tokenizer(
        text, max_length=128, truncation=True,
        padding='max_length', return_tensors='pt'
    )
    with torch.no_grad():
        logits = model(
            input_ids=enc['input_ids'].to(device),
            attention_mask=enc['attention_mask'].to(device)
        ).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    top   = np.argsort(probs)[::-1][:top_k]
    return [(le.classes_[i], round(float(probs[i]), 3)) for i in top]


test_cases = [
    'A detective hunts a serial killer leaving cryptic clues across the city.',
    'Two best friends accidentally swap lives at their high school reunion.',
    'Scientists reveal secrets of deep ocean creatures never seen before.',
    'A family moves into an old house and discovers a terrifying supernatural presence.',
    'An elite spy must stop a global terrorist plot before midnight.',
    'A young girl discovers a magical kingdom and befriends a dragon.',
    # ลองใส่ description ของตัวเองได้เลย!
]

print('=' * 60)
for text in test_cases:
    preds = predict_genre(text, model, tokenizer, le, device)
    print(f'Input: {text[:60]}...')
    for genre, prob in preds:
        bar = '█' * int(prob * 30)
        print(f'  {genre:<28} {bar} {prob:.3f}')
    print()


## 8. Save Model

In [ ]:
# Save tokenizer + model สำหรับใช้ต่อ
model.save_pretrained('./netflix_bert_classifier')
tokenizer.save_pretrained('./netflix_bert_classifier')
print('Model saved to ./netflix_bert_classifier/')

# หรือ download ออกมา
# from google.colab import files
# import shutil
# shutil.make_archive('netflix_bert_classifier', 'zip', './netflix_bert_classifier')
# files.download('netflix_bert_classifier.zip')


## 📊 Summary

| Step | Detail |
|---|---|
| Model | DistilBERT (distilbert-base-uncased) |
| Parameters | ~66M |
| Max Length | 128 tokens |
| Batch Size | 32 |
| Epochs | 4 |
| Optimizer | AdamW (lr=2e-5) |
| Scheduler | Linear warmup (10%) |

### ทำไม BERT ถึงแม่นกว่า TF-IDF?

- **Contextual embeddings** — BERT เข้าใจความหมายของคำตาม context เช่น "dark" ใน horror vs drama
- **Pre-trained knowledge** — ถูก train บน Wikipedia + BookCorpus มาแล้ว เข้าใจภาษาอังกฤษลึกกว่า
- **Subword tokenization** — จัดการคำใหม่หรือชื่อเฉพาะได้ดีกว่า TF-IDF

### ข้อจำกัด

- ต้องใช้ GPU (ช้ามากบน CPU)
- Dataset ยังเล็กและ imbalanced (Drama 2,427 vs Horror 163)
- Description สั้นมาก ทำให้แม้แต่ BERT ก็ยังสับสน Drama vs Comedy ได้
